In [2]:
# Model tuning of the machine learning 
# Sibusiso Mathebula

In [3]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [4]:
df_original= pd.read_csv("diabetes (1).csv")
print(df_original.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None


In [5]:
def create_features(X):
        X = X.copy()
        
        X['PregnancyGroup'] = pd.cut(
            X['Pregnancies'],
        bins=[-1,0,3,6,20],
        labels=['0','1-3','4-6','7+'])
        
        X['AgeGroup'] = pd.cut(
            X['Age'],
            bins=[20,30,40,50,60,80],
            labels=['20-30','31-40','41-50','51-60','61-80']
        )
        
        X['Glucose_BMI'] = X['Glucose']*X['BMI']

        return X
        
feature_creator = FunctionTransformer(create_features)




In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [7]:
models = {
       "Logistic Regression": LogisticRegression(max_iter=1000),
       "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(),
        "SVM": SVC(),
       "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    }

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
    
X = df_original.drop("Outcome", axis=1)
y = df_original["Outcome"]
    
X_train, X_test, y_train, y_test = train_test_split(
       X, y, test_size=0.2, random_state=42, stratify=y
    )
    
engineer_cols = ['Pregnancies','Age','BMI','Glucose']
num_features = X.select_dtypes(['int64','float64']).columns.tolist()
cat_features = X.select_dtypes(['object','category']).columns.tolist()
    
    
numeric_pipeline = Pipeline([
       ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
categorical_pipeline = Pipeline([
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
       
    ])
    
feature_engineer_cols =  ['AgeGroup','PregnancyGroup']
    
feature_pipeline=  Pipeline([
           ('feature_creation', feature_creator)])
    
preprocessor = Pipeline([
    ('feature_creation', feature_creator),
    ('column_transformer', ColumnTransformer(
        transformers=[
            ('num', numeric_pipeline, num_features),
            ('cat', categorical_pipeline, cat_features)
        ],
        remainder='drop'
    ))
])
    
results = []

for name, model in models.items():
    
        pipeline = Pipeline([
          ('preprocessor', preprocessor),
          ('model', model)
              ])
    
        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
    
        results.append({
        "Model": name,
          "Accuracy": accuracy_score(y_test, y_pred),
          "Precision": precision_score(y_test, y_pred),
          "Recall": recall_score(y_test, y_pred)
       })
    
results_df = pd.DataFrame(results)
print(results_df)

c:\Users\sibus\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:23:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


                 Model  Accuracy  Precision    Recall
0  Logistic Regression  0.714286   0.608696  0.518519
1        Decision Tree  0.720779   0.627907  0.500000
2        Random Forest  0.740260   0.640000  0.592593
3                  SVM  0.753247   0.660000  0.611111
4              XGBoost  0.733766   0.622642  0.611111


In [9]:
from sklearn.metrics import roc_auc_score
score = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:,1])
    
print(f'The roc_auc score is {score}')

The roc_auc score is 0.8051851851851852
